In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
print("✅ Imports done")

✅ Imports done


In [3]:
import sys
print(sys.executable)

d:\Desktop\ENGG\E3_Mini_Project\Building_Energy_Consumption_Predictor\.venv\Scripts\python.exe


In [4]:
train = pd.read_feather('../data/train.feather')
building = pd.read_feather('../data/building_metadata.feather')
weather = pd.read_feather('../data/weather_train.feather')

print("TRAIN:", train.shape)
print("BUILDING:", building.shape)
print("WEATHER:", weather.shape)

TRAIN: (20216100, 4)
BUILDING: (1449, 6)
WEATHER: (139773, 9)


In [5]:
print(train.info())
print(weather.info())
print(building.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20216100 entries, 0 to 20216099
Data columns (total 4 columns):
 #   Column         Dtype         
---  ------         -----         
 0   building_id    int16         
 1   meter          int8          
 2   timestamp      datetime64[ns]
 3   meter_reading  float32       
dtypes: datetime64[ns](1), float32(1), int16(1), int8(1)
memory usage: 289.2 MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139773 entries, 0 to 139772
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   site_id             139773 non-null  int8          
 1   timestamp           139773 non-null  datetime64[ns]
 2   air_temperature     139718 non-null  float32       
 3   cloud_coverage      70600 non-null   float32       
 4   dew_temperature     139660 non-null  float32       
 5   precip_depth_1_hr   89484 non-null   float32       
 6   sea_level_pressure  

In [6]:
print("Train memory (MB):", train.memory_usage().sum() / 1024**2)
print("Weather memory (MB):", weather.memory_usage().sum() / 1024**2)

Train memory (MB): 289.1937561035156
Weather memory (MB): 4.9321489334106445


In [8]:
df = train.merge(building, on="building_id", how="left")
df = df.merge(weather, on=["site_id", "timestamp"], how="left")

print(df.shape)

(20216100, 16)


In [9]:
df = train.merge(building, on="building_id", how="left")
df = df.merge(weather, on=["site_id", "timestamp"], how="left")

print("Merged shape:", df.shape)
print("Memory (MB):", df.memory_usage().sum() / 1024**2)

Merged shape: (20216100, 16)
Memory (MB): 1098.9365730285645


In [10]:
df.drop(columns=["floor_count"], inplace=True)

In [11]:
weather_cols = [
    "air_temperature",
    "cloud_coverage",
    "dew_temperature",
    "precip_depth_1_hr",
    "sea_level_pressure",
    "wind_direction",
    "wind_speed"
]

for col in weather_cols:
    df[col] = df.groupby("site_id")[col].transform(
        lambda x: x.fillna(x.median())
    )

In [13]:
df["meter_reading_log"] = np.log1p(df["meter_reading"])

In [14]:
df["hour"] = df["timestamp"].dt.hour.astype("int8")
df["month"] = df["timestamp"].dt.month.astype("int8")
df["weekday"] = df["timestamp"].dt.weekday.astype("int8")
df["is_weekend"] = df["weekday"].isin([5,6]).astype("int8")

In [15]:
df["primary_use"] = df["primary_use"].cat.codes.astype("int8")

In [16]:
df["energy_per_sqft"] = df["meter_reading"] / df["square_feet"]
df["energy_per_sqft"] = df["energy_per_sqft"].astype("float32")

In [17]:
df.drop(columns=["timestamp"], inplace=True)

In [18]:
print("Final memory usage (MB):", df.memory_usage().sum() / 1024**2)
df.info()

Final memory usage (MB): 1098.935920715332
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20216100 entries, 0 to 20216099
Data columns (total 20 columns):
 #   Column              Dtype  
---  ------              -----  
 0   building_id         int16  
 1   meter               int8   
 2   meter_reading       float32
 3   site_id             int8   
 4   primary_use         int8   
 5   square_feet         int32  
 6   year_built          float32
 7   air_temperature     float32
 8   cloud_coverage      float32
 9   dew_temperature     float32
 10  precip_depth_1_hr   float32
 11  sea_level_pressure  float32
 12  wind_direction      float32
 13  wind_speed          float32
 14  hour                int8   
 15  month               int8   
 16  weekday             int8   
 17  is_weekend          int8   
 18  meter_reading_log   float32
 19  energy_per_sqft     float32
dtypes: float32(11), int16(1), int32(1), int8(7)
memory usage: 1.1 GB


In [19]:
df["year_built"].isnull().mean()

np.float64(0.5999003269671203)

In [20]:
df.drop(columns=["year_built"], inplace=True)

In [23]:
df.drop(columns=["meter_reading"], inplace=True)

In [25]:
df["square_feet"].max()

np.int32(875000)

In [26]:
print("Final memory usage (MB):", df.memory_usage().sum() / 1024**2)

Final memory usage (MB): 944.6993179321289


In [27]:
df.memory_usage(deep=True).sum() / 1024**2

np.float64(944.6993179321289)

In [28]:
df.to_feather("../data/cleaned_data.feather")